In [ ]:
from pathlib import Path

import numpy as np
from skimage.io import imread
from scipy.ndimage import label
from calmutils.imageio.tiff_imagej import get_imagej_tiff_pixel_size
from tqdm.notebook import tqdm

from segmentation_utils import threshold_segmentation, edt_watershed_instance_segmentation
from visualization_utils import get_segmentation_visualization
from io_helpers import imsave_nowarnings

In [ ]:
# path containing files to visualize
in_path = '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_2'


# subdirectory containing input data
# leave empty ('') if image files are directly in in_path
in_subdirectory = 'patches_gfp+'
in_file_pattern = '[!.]*_ch0*.tif'

# path to which output is saved
# default: put results in subdirectory called 'segmentation-threshold'
out_subdirectory = 'patches-segmentation-threshold'

## parameters for segmentation
# sigma of blur to apply before thresholding
blur_sigma = 1
# size (in pixels) of small objects/small holes to discard
small_hole_size = 100
small_hole_size_perplane = 50
small_object_size = 200
# radius of binary closing applied to mask (can be slow, set to 0 to skip)
closing_radius = 0

# do instance segmentation?
# can be: None/False - don't do instance segmentation
# 'connected-components' - only do connected components labelling
# 'watershed' - do watershed transform on edt of mask
instance_segmentation = None
# if instance segmentation by watershed, what is the required prominence of EDT maxima to be considered as seed points
# lower values: oversegmentation, higher values: undersegmentation
h_maxima_threshold = 1.5

# whether to save a simple png visualization of segmentation results in a subfolder
save_visualization = True

In [ ]:
in_path = Path(in_path)
out_path = in_path / out_subdirectory

# get all files matching pattern in in_path
in_files = sorted((in_path / in_subdirectory).glob(in_file_pattern))

# show for verification
in_files

In [ ]:
# make output directories if necessary
if not out_path.exists():
    out_path.mkdir(parents=True)

visualization_path = out_path / 'quick_result_visualization'
if save_visualization and not visualization_path.exists():
    visualization_path.mkdir(parents=True)

for in_file in tqdm(in_files):
    
    img = imread(in_file)

    # try to get pixel size, default to [1,1,..] if no metadata
    try:
        pixel_size = get_imagej_tiff_pixel_size(in_file)
    except ValueError:
        pixel_size = [1] * img.ndim
    
    # run segmentation
    segmented = threshold_segmentation(img, blur_sigma, small_hole_size, small_hole_size_perplane, small_object_size, closing_radius)

    # do instance segmentation / labelling if necessary
    if not instance_segmentation:
        pass
    elif instance_segmentation == 'watershed':
        segmented = edt_watershed_instance_segmentation(segmented, h_maxima_threshold, pixel_size)
    elif instance_segmentation == 'connected-components':
        segmented = label(segmented)
    else:
        raise ValueError(f'instance segmentation method "{instance_segmentation}" not available')

    # convert to 8 or 16 bit as necessary
    segmented = segmented.astype(np.uint16) if segmented.max() > 255 else segmented.astype(np.uint8)

    # make filepath for output
    outfile = out_path / (in_file.stem + '_segmented.tif')
    imsave_nowarnings(outfile, segmented)

    if save_visualization:
        visualization_projection = get_segmentation_visualization(segmented, img)
        # make filepath for output
        outfile_visualization = visualization_path / (in_file.stem + '_segmented_projection.png')            
        imsave_nowarnings(str(outfile_visualization), visualization_projection)
        

In [ ]:
from matplotlib import pyplot as plt
import napari

viewer = napari.current_viewer() or napari.Viewer()

viewer.add_image(img)
viewer.add_labels(segmented)


plt.imshow(visualization_projection)